# ClauseLens Fine-Tuning (Qwen2.5-1.5B-Instruct, Unsloth, LoRA)

This notebook fine-tunes on the ClauseLens dataset in the exact format used by training/inference (system/user/assistant messages). Train with safetensors, merge LoRA, convert to GGUF for local inference with llama-cpp-python.

In [ ]:
# ==============================================================================
# 0. INSTALL
# ==============================================================================
!pip install -q -U unsloth
!pip install -q datasets transformers trl accelerate bitsandbytes huggingface_hub
!pip install -q "xformers<0.0.27" --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import os
import torch
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

In [ ]:
# ==============================================================================
# 2. CONFIG
# ==============================================================================
# Base model (SAFETENSORS for training)
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"

# Dataset - upload train.jsonl from ClauseLens/data/dataset/ to Colab
DATASET_PATH = "/content/train.jsonl"

# Output
OUTPUT_DIR = "/content/qwen2.5-1.5b-clauselens-lora"
HF_REPO = "vikram2xx7/Qwen2.5-1.5B-Instruct-ClauseLens"  # change if needed

# Training (6GB VRAM optimized)
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4

In [ ]:
# ==============================================================================
# 3. CHECK GPU
# ==============================================================================
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
# ==============================================================================
# 4. LOAD CLAUSELENS DATASET (messages format)
# ==============================================================================
data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f"Loaded {len(data)} examples")

# Convert messages to text format for SFTTrainer with dataset_text_field
# For Qwen2.5 chat template, we can use tokenizer.apply_chat_template or just use the messages
# Unsloth works well with the chat format
texts = []
for item in data:
    messages = item['messages']  # system, user, assistant
    # Build text in Qwen chat format
    text = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    texts.append(text)

dataset = Dataset.from_dict({"text": texts})
print("Dataset ready:", dataset)
print("\nExample:\n", dataset[0]['text'][:400] + "...")

In [ ]:
# ==============================================================================
# 5. LOAD MODEL
# ==============================================================================
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    dtype = None,
)

In [ ]:
# ==============================================================================
# 6. LORA
# ==============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# ==============================================================================
# 7. TRAIN
# ==============================================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        warmup_ratio = 0.03,
        logging_steps = 10,
        save_steps = 200,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        seed = 3407,
        report_to = "none",
        remove_unused_columns = False,
    ),
)

trainer.train()

In [ ]:
# ==============================================================================
# 8. SAVE LORA
# ==============================================================================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# ==============================================================================
# 9. PUSH TO HF (optional)
# ==============================================================================
notebook_login()
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

## Next steps (local)

1. Download the LoRA adapter from HF or from Colab (zip it)
2. Merge LoRA with base model `unsloth/Qwen2.5-1.5B-Instruct` (safetensors)
3. Convert merged model to GGUF with `llama-gguf` / `llama.cpp` quantization (use `IQ2_XS` or `Q2_K_M` for 6GB VRAM)
4. Drop the `.gguf` file into `ClauseLens/models/finetuned/` as `clause-review-qwen2.5-3b-q4_k_m.gguf` (or update `DEFAULT_MODEL_PATH` in `core/ft_analyzer.py`)
5. Test with `FineTunedAnalyzer` - it uses the shared prompt/schema and has the verifier gate enabled